# Report on "Neural decoding of competitive decision-making in Rock-Paper-Scissors"

Team Members: Emma Feege, Felipe Potenza, Radu-Mihai Savancea

## Experiment Reproduction

### Preprocessing

To replicate the original preprocessing pipeline, we implemented a stepwise workflow in Python using MNE and MNE-BIDS, closely following the procedures of the MATLAB FieldTrip script. The preprocessing procedure for each participant pair consisted of the following stages:



1. Data loading: Raw EEG data were stored in BIDS-compatible `.bdf` files for each session. Data were loaded into memory using mne_bids.read_raw_bids, preserving the original sampling rate of 2048 Hz. Each pair contained two participants’ data in a single file, and channels were selected and separated based on participant-specific prefixes.

```python
# Load file
def get_from(bids_root: pathlib.Path) -> "BidsDataset":
    subjects = []
    tsv_path = bids_root / "participants.tsv"
    with open(tsv_path, newline="") as tsvfile:
        reader = csv.DictReader(tsvfile, delimiter="\t")
        for row in reader:
            # Excluded subjects matching the MATLAB script's exclusions: 10, 23, 24
            if row["participant_id"] not in ["sub-10", "sub-23", "sub-24"]:
                subjects.append(Subject.from_tsv_row(bids_root, row))
    return BidsDataset(bids_root, subjects)
```

```python
# Load each player and subject
def from_tsv_row(bids_root, row: Mapping[Any]) -> "Subject":
    pid = row["participant_id"]
    # --- Parsing Player Metadata ---
    p1_bad = [ch.strip() for ch in row["player1_pre_processing_channels_fixed"].split(",") if ch.strip()]
    player1 = Player(Gender(row["player1_gender"]), int(row["player1_age"]), Handedness(row["player1_handedness"]), p1_bad)
    p2_bad = [ch.strip() for ch in row["player2_pre_processing_channels_fixed"].split(",") if ch.strip()]
    player2 = Player(Gender(row["player2_gender"]), int(row["player2_age"]), Handedness(row["player2_handedness"]), p2_bad)
    events = get_events_for_subject(bids_root, pid)
    return Subject(pid, player1, player2, events)
```

2. Channel renaming and montage assignment: Channels in the original data used a custom BioSemi naming convention (A1-A32, B1-B32) with prefixes indicating player assignment. These were mapped to standard 10–20 labels using a BioSemi-to-10-20 mapping derived from the MATLAB `biosemi64.lay` layout. A custom 3D electrode montage was created from the `biosemi64.mat` file to replicate the spatial arrangement used in the MATLAB preprocessing. The montage was applied to each participant’s dataset to set accurate electrode positions.

```python
# --- Global Constants & Setup (Mimicking FieldTrip Layout/Geometry Loading) ---
# The standard BioSemi codes in order (A1-A32 then B1-B32) corresponding 
# to the channel sequence used by FieldTrip's biosemi64.lay template.
BIOSEMI_ORDERED_CODES = [
    'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 
    'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32',
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 
    'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32'
]

# Get the standard 10-20 names from MNE's built-in montage (64 channels)
montage_1020_ref = mne.channels.make_standard_montage('standard_1020')
TEN_TWENTY_ORDERED_LABELS = montage_1020_ref.ch_names[:64]

# CRITICAL MAPPING: BioSemi Code -> 10-20 Label.
# COMPARISON (MATLAB): This is equivalent to loading the labels from 'biosemi64.lay'.
BIOSEMI_CODE_TO_1020_LABEL = dict(zip(BIOSEMI_ORDERED_CODES, TEN_TWENTY_ORDERED_LABELS))
```

```python
# Helper function to rename, apply map, and set montage (The critical function)
def fix_names(inst, prefix):
    # 1. Strip player prefix ('2-A1' -> 'A1')
    rename_map_prefix = {ch: ch.replace(prefix, "") for ch in inst.ch_names}
    inst.rename_channels(rename_map_prefix)
    
    # 2. Rename using the dynamically generated map (from .lay equivalent)
    final_map = {k: v for k, v in BIOSEMI_CODE_TO_1020_LABEL.items() if k in inst.ch_names}
    if final_map:
        inst.rename_channels(final_map) 
        # COMPARISON (MATLAB): Equivalent to 'data_epoch.label(1:64) = layout.label(1:64);'
    
    # 3. Set channel types and pick only the final EEG channels
    inst.set_channel_types({ch: 'eeg' for ch in inst.ch_names}, verbose=False) 
    eeg_chans = [ch for ch in inst.ch_names if ch in FULL_MNE_BISEOMI_MONTAGE.ch_names]
    inst.pick_channels(eeg_chans, ordered=True, verbose=False) 
    
    # Use the canonical list of channels from the 10-20 system (64 channels)
    montage_1020 = mne.channels.make_standard_montage("standard_1020")
    
    # Drop any non-EEG/unmapped channels (e.g., EOGs, references)
    eeg_chans = [ch for ch in inst.ch_names if ch in montage_1020.ch_names]
    inst.pick_channels(eeg_chans, ordered=True, verbose=False)
    
    # 4. Apply Montage (sets the 3D coordinates from the .mat file)
    inst.set_montage(FULL_MNE_BISEOMI_MONTAGE, match_case=False, verbose=False)
    # COMPARISON (MATLAB): This links the new 10-20 channel names to the 3D coordinates from 'biosemi64.mat'.
```

3. Bad channel interpolation: Channels identified as noisy in a participant-specific metadata file were marked as bad and interpolated using MNE’s distance-based interpolation method, analogous to MATLAB’s ft_channelrepair. Neighboring electrodes were determined automatically using the 3D montage coordinates.

```python
def _interpolate(self, raw, player_meta, label):
      """Handles bad channel interpolation (MATLAB's ft_channelrepair)."""
      bads = [ch for ch in player_meta.preprocessing_channels_fixed if ch in raw.ch_names]
      if bads:
          raw.info['bads'] = bads
          raw.interpolate_bads(reset_bads=True, verbose=False)
          print(f"  {label}: Interpolated {bads}")
```

4. Downsampling: The EEG data were downsampled to 256 Hz to reduce computational load, preserving temporal structure for subsequent analyses. This corresponds to the MATLAB pipeline’s resampling step.

```python
raw_p1.resample(256, verbose=False)
raw_p2.resample(256, verbose=False)
```

5. Epoching: Continuous EEG data were segmented into epochs relative to the onset of the decision phase of each trial. Epochs spanned from -0.2 s pre-stimulus to 5 s post-stimulus. Event information was read from the BIDS event files, ensuring alignment with the original trial structure.



```python
def _create_epochs(self, raw):
    """Converts continuous data into segmented trials (MATLAB's ft_preprocessing with cfg.trl)."""
    onset_times = [e.onset for e in self.events]
    annot = mne.Annotations(onset=onset_times, duration=[0]*len(onset_times), description=['trial_start']*len(onset_times))
    raw.set_annotations(annot)
    # Create event array from annotations
    events, _ = mne.events_from_annotations(raw, verbose=False)
    # Epoching (-0.2s pre-stimulus, 5.0s post-stimulus)
    return mne.Epochs(raw, events, tmin=-0.2, tmax=5.0, baseline=(-0.2, 0), preload=True, verbose=False)
```

6. Saving preprocessed data: Preprocessed epochs for each participant were saved as `.fif` files, maintaining all channel metadata and montage information. Separate files were created for each participant within a pair to support independent downstream analyses.



```python
def _save(self, ep1, ep2, output_dir):
    """Saves the final Epochs objects to disk (MATLAB's save function)."""
    pair_num = self.id.replace('sub-', '')
    out_folder = output_dir / "derivatives"
    out_folder.mkdir(parents=True, exist_ok=True)
    
    p1_fname = out_folder / f"pair-{pair_num}_player-1_task-RPS_eeg_epo.fif"
    p2_fname = out_folder / f"pair-{pair_num}_player-2_task-RPS_eeg_epo.fif"
    
    ep1.save(p1_fname, overwrite=True, verbose=False)
    ep2.save(p2_fname, overwrite=True, verbose=False)
```

This replication pipeline preserved the structure and processing order of the original MATLAB implementation, including participant-specific channel selection, bad channel interpolation, downsampling, and epoching. By mirroring the original FieldTrip workflow in Python/MNE, the preprocessing ensures consistency across datasets and allows for systematic comparison with subsequent analyses.

### Decoding

### Markov Chain Analysis

To assess how predictable each participant’s behavior was during the Rock–Paper–Scissors game, the authors applied a Markov chain analysis to the behavioral response data. The goal of this analysis was not to model an "optimal play", but to quantify the extent to which a participant’s next response could be predicted from their previous responses. The Prediction accuracy was used as a direct measure of behavioral predictability.

The analysis was conducted separately for each participant. For every trial in the experiment, the authors tracked the participant’s responses across time and computed how often each response (Rock, Paper, or Scissors) followed a specific response on the previous trial. Using this cumulative information, the authors constructed transition probabilities describing the likelihood of playing Rock, Paper, or Scissors given the response on the previous trial. These probabilities formed a 3 × 3 transition matrix, where each row corresponded to the previous response and each column corresponded to the predicted next response.

For each trial, the model predicted the most likely next response based on the previous valid response and the transition probabilities estimated from the selected window. The Prediction accuracy was calculated as the proportion of trials for which the predicted response matched the participant’s actual response, excluding missing responses.

An accuracy of 33.3% reflects chance-level performance, whereas accuracies above chance indicate systematic, history-dependent behavior. These accuracy values provided a quantitative measure of behavioral predictability that could be compared across participants and window sizes and related to neural decoding results. While this Markov approach captures local, pairwise dependencies between trials in an interpretable manner, more flexible deep learning time-series models could potentially capture higher-order and longer-range temporal patterns in future work.

### Replication Challenges

Replicating the experimental pipeline described in the original EEG study proved to be challenging due to several practical and methodological issues encountered during dataset handling and preprocessing. First, the size of the dataset (approximately 80 GB) posed significant computational and storage challenges, increasing the time required for data transfer, loading, and preprocessing. Such large-scale data handling requirements were not explicitly discussed in the original paper, yet they represent a non-trivial barrier to replication.

Additionally, although the dataset is nominally organized according to the BIDS standard, the structure is incomplete. In particular, the required `_channels.tsv` files were missing for each session. This omission complicates automated preprocessing pipelines and requires manual intervention to reconstruct channel metadata, undermining the advantages of standardized data organization.

Further difficulties arose from the provided MATLAB preprocessing code. The code was poorly annotated, making it difficult to interpret key preprocessing steps and design choices. This lack of documentation significantly increased the effort required to understand and reimplement the pipeline in a different software ecosystem. Moreover, the authors use a custom electrode naming convention, which is not aligned with standard EEG montages. As a result, manual mapping between electrode names was required to ensure compatibility with commonly used EEG analysis libraries.

A particularly critical issue concerned the electrode spatial information. The authors rely on a custom three-dimensional coordinate system for electrode positions, yet provide no description of the coordinate space, measurement units, or scaling. The MATLAB library used in the original implementation automatically rescales electrode positions so that the head representation forms a unit circle. This behavior is not replicated in the MNE-Python library used in our reimplementation. In the absence of documentation, the scaling factor had to be inferred empirically, introducing uncertainty into the spatial accuracy of the electrode layout.

Data formatting inconsistencies further hindered replication. Several `.tsv` files use non-standard formatting, where columns are separated by variable numbers of spaces rather than special character, such as the use of commas, and missing values are represented by blank spaces. This deviates from standard TSV conventions and breaks many parsing tools, requiring custom preprocessing scripts to correctly load the data.

Finally, the paper does not provide any visualizations of the EEG signals after preprocessing. The absence of qualitative references, such as example time series, power spectra, or topographical plots, makes it impossible to visually compare our preprocessed data with the authors’ results. This lack of intermediate validation outputs limits confidence in the correctness of the replicated preprocessing pipeline, even when numerical results appear reasonable.

Notes about things we identified during the code development (preprocessing) --> you can decide whether it's relevant to add to the essay:
- Dificulties with the size of the dataset (~80 GB);
- Dataset BIDS structure is incomplete --> it's missing the "_channels.tsv" file for each session; 
- Poorly annotated matlab code;
- They use their own electrode naming conevention --> we had to map manually map it;
- They have their own 3D coordinate map for the electrodes, but there are not descriptions on how it works (e.g., measurement unit)
   - the library they use in matlab does automatic scaling so that the shape of the electrodes in the head is a unit circle --> that function does not exist in MNE library --> without any descriptions, we have to guess what the scale is to get a unit circle;
- .tsv files use a _horrible_ convetion of spaces to separate columns, and if a column doesn't contain a value, it's represent by a blank space;
- They don't have a single image showing what the eeg looks like after the preprocessing --> it's impossible for us to compare our results to theirs;